# Task 3: The Smoking Gun
## Explainable AI with SHAP for DistilBERT-LoRA Stylometry Detection

**Objective:** Use SHAP (SHapley Additive exPlanations) to interpret the fine-tuned DistilBERT-LoRA model and identify linguistic "smoking guns" that reveal AI authorship

**Prerequisites:** Complete **Task 2.3 (Tier C)** first to train the LoRA model!

**Components:**
1. Load fine-tuned DistilBERT-LoRA model
2. SHAP analysis on AI samples (word-level attribution)
3. Error analysis (False Positives & Negatives)
4. Identify common AI markers

---

**Why Explainable AI?**
- Understand **what** the model learned
- Identify **which words** reveal AI authorship
- Find **linguistic patterns** that distinguish Human vs AI
- Build trust through interpretability

---

## What This Reveals:
- 🔍 Word-level contributions to predictions
- ⚠️ Common "robotic" markers (delve, tapestry, intricate)
- ✅ Successfully detected AI patterns
- ❌ Misclassifications and edge cases


## 1. Setup and Installation

In [ ]:
# Install required packages
!pip install transformers peft shap datasets torch pandas numpy scikit-learn matplotlib seaborn -q

print("✅ All packages installed successfully!")

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import torch
from typing import List, Dict
import warnings
warnings.filterwarnings('ignore')

# Hugging Face & PEFT
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel
from datasets import Dataset

# SHAP for explainability
import shap

# Evaluation
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.model_selection import train_test_split
from tqdm import tqdm

print("✅ Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"SHAP version: {shap.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. Configuration

**IMPORTANT:** Update these paths to match your setup!
- Model directory should contain the trained LoRA adapter from Tier C
- CSV files should be the same ones used for training

In [ ]:
# Paths
MODEL_DIR = "/content/lora_adapter"  # Update to your LoRA adapter location
BASE_MODEL = "distilbert-base-uncased"
OUTPUT_DIR = "/content/xai_analysis"

# CSV file paths (same as Tier C training)
CSV_PATHS = {
    "Class 1 (Human)": {
        "path": "/content/precog.csv",
        "text_column": "text",
        "label": "Human"
    },
    "Class 2 (AI)": {
        "path": "/content/class_2_combined.csv",
        "text_column": "text",
        "label_column": "author_target"
    },
    "Class 3 (AI Mimic)": {
        "path": "/content/class_3_combined.csv",
        "text_column": "text",
        "label_column": "author_target"
    }
}

# SHAP configuration
NUM_IMPOSTER_SAMPLES = 5  # Number of AI samples to analyze with SHAP
NUM_ERROR_SAMPLES = 3     # Number of errors to show in detail
MAX_LENGTH = 512
TEST_SIZE = 0.2
SEED = 42

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("✅ Configuration complete!")
print(f"   Model directory: {MODEL_DIR}")
print(f"   Output directory: {OUTPUT_DIR}")
print(f"   Device: {device}")

## 3. Load Fine-Tuned LoRA Model

In [ ]:
print("=" * 80)
print("LOADING FINE-TUNED MODEL")
print("=" * 80)

# Load tokenizer
print(f"\n📂 Loading tokenizer from: {MODEL_DIR}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
print(f"   ✓ Tokenizer loaded")

# Load base model
print(f"\n📂 Loading base model: {BASE_MODEL}")
base_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=2,
    id2label={0: "Human", 1: "AI"},
    label2id={"Human": 0, "AI": 1}
)
print(f"   ✓ Base model loaded")

# Load LoRA adapter
print(f"\n📂 Loading LoRA adapter from: {MODEL_DIR}")
model = PeftModel.from_pretrained(base_model, MODEL_DIR)
print(f"   ✓ LoRA adapter loaded")

# Move to device and set to evaluation mode
model = model.to(device)
model.eval()

print(f"\n✅ Model ready for inference!")
print(f"   Device: {device}")
print(f"   Mode: Evaluation")

## 4. Load Test Data

Use the same train-test split as Tier C training

In [ ]:
print("\n" + "=" * 80)
print("LOADING TEST DATA")
print("=" * 80)

# Load all CSV files
all_dfs = []

for class_name, config in CSV_PATHS.items():
    try:
        print(f"\n📂 Loading {class_name}: {config['path']}")
        df = pd.read_csv(config['path'])
        
        # Set label
        if 'label_column' in config and config['label_column'] in df.columns:
            df['label'] = df[config['label_column']]
        else:
            df['label'] = config.get('label', 'Unknown')
        
        # Keep only text and label
        df = df[[config['text_column'], 'label']].copy()
        df.columns = ['text', 'label']
        
        print(f"   ✓ Loaded {len(df)} samples")
        all_dfs.append(df)
        
    except FileNotFoundError:
        print(f"   ✗ File not found: {config['path']}")
    except Exception as e:
        print(f"   ✗ Error: {e}")

# Combine and create binary labels
df_combined = pd.concat(all_dfs, ignore_index=True)
df_combined['binary_label'] = df_combined['label'].apply(
    lambda x: 0 if x.lower() == 'human' else 1
)

# Split (same as training)
_, test_df = train_test_split(
    df_combined[['text', 'label', 'binary_label']], 
    test_size=TEST_SIZE, 
    random_state=SEED, 
    stratify=df_combined['binary_label']
)

test_labels_original = test_df['label'].values

print(f"\n✅ Test set loaded: {len(test_df)} samples")
print(f"\n📊 Label distribution:")
print(f"  Human (0): {(test_df['binary_label'] == 0).sum()} samples")
print(f"  AI (1):    {(test_df['binary_label'] == 1).sum()} samples")

## 5. Create Prediction Function for SHAP

SHAP needs a function that takes texts and returns AI probabilities

In [ ]:
def predict_fn(texts: List[str]) -> np.ndarray:
    """
    Predict AI probability for a list of texts.
    
    Args:
        texts: List of text strings
        
    Returns:
        Array of AI probabilities (shape: [batch_size,])
    """
    # Tokenize
    encodings = tokenizer(
        texts,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=True,
        return_tensors='pt'
    )
    
    # Move to device
    encodings = {k: v.to(device) for k, v in encodings.items()}
    
    # Get predictions
    with torch.no_grad():
        outputs = model(**encodings)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)
        ai_probs = probs[:, 1].cpu().numpy()  # AI class probability
    
    return ai_probs

print("✅ Prediction function created!")

# Test it
test_texts = ["The quick brown fox jumps over the lazy dog.", "AI-generated content often sounds formulaic."]
test_probs = predict_fn(test_texts)
print(f"\n📝 Test predictions:")
for text, prob in zip(test_texts, test_probs):
    print(f"   '{text[:50]}...' → AI prob: {prob:.4f}")

## 6. Select AI Samples for SHAP Analysis

Choose a few AI samples that were correctly classified

In [ ]:
print("\n" + "=" * 80)
print("SELECTING AI SAMPLES FOR SHAP ANALYSIS")
print("=" * 80)

# Select AI samples
ai_samples = test_df[test_df['binary_label'] == 1].copy()

if len(ai_samples) < NUM_IMPOSTER_SAMPLES:
    print(f"⚠️  Only {len(ai_samples)} AI samples available")
    selected_samples = ai_samples
else:
    selected_samples = ai_samples.sample(n=NUM_IMPOSTER_SAMPLES, random_state=SEED)

selected_texts = selected_samples['text'].tolist()
selected_labels = selected_samples['label'].tolist()

print(f"\n🎭 Selected {len(selected_texts)} AI samples for analysis")

# Verify predictions
print(f"\n🔮 Verifying predictions...")
predictions = predict_fn(selected_texts)

for i, (text, label, pred) in enumerate(zip(selected_texts, selected_labels, predictions)):
    text_preview = text[:80] + "..." if len(text) > 80 else text
    print(f"\n   Sample {i+1}:")
    print(f"      Label: {label}")
    print(f"      AI Probability: {pred:.4f}")
    print(f"      Text: {text_preview}")

## 7. Create SHAP Explainer

This may take several minutes as SHAP computes feature attributions

In [ ]:
print("\n" + "=" * 80)
print("CREATING SHAP EXPLAINER")
print("=" * 80)
print("\n⏳ This may take a few minutes...\n")

# Create text masker
masker = shap.maskers.Text(tokenizer=tokenizer)

# Create explainer
explainer = shap.Explainer(predict_fn, masker=masker)

print("✅ SHAP explainer created!")

## 8. Compute SHAP Values

Calculate word-level attributions for each selected sample

In [ ]:
print("\n" + "=" * 80)
print("COMPUTING SHAP VALUES")
print("=" * 80)
print(f"\n⏳ Analyzing {len(selected_texts)} samples...")
print("   (This will take several minutes...)\n")

shap_values_list = []

for i, text in enumerate(tqdm(selected_texts, desc="Computing SHAP values")):
    try:
        shap_values = explainer([text])
        shap_values_list.append(shap_values)
        
    except Exception as e:
        print(f"   ⚠️  Error computing SHAP for sample {i+1}: {e}")
        shap_values_list.append(None)

print(f"\n✅ SHAP computation complete!")

## 9. Visualize SHAP Results

Show word-level attributions for each sample

In [ ]:
print("\n" + "=" * 80)
print("SHAP ANALYSIS RESULTS")
print("=" * 80)

for i, (shap_values, label, pred) in enumerate(zip(shap_values_list, selected_labels, predictions)):
    if shap_values is None:
        print(f"\n❌ Sample {i+1}: SHAP computation failed")
        continue
    
    print(f"\n{'=' * 80}")
    print(f"SAMPLE {i+1}: {label}")
    print(f"AI Probability: {pred:.4f}")
    print(f"{'=' * 80}")
    
    # Get token values
    token_values = shap_values.values[0]
    tokens = shap_values.data[0]
    
    # Get top positive contributors (words pushing toward AI)
    token_importance = list(zip(tokens, token_values))
    token_importance_sorted = sorted(token_importance, key=lambda x: abs(x[1]), reverse=True)
    
    print(f"\n🎯 Top 10 words contributing to AI detection:")
    for j, (token, value) in enumerate(token_importance_sorted[:10]):
        direction = "→ AI" if value > 0 else "→ Human"
        print(f"   {j+1:2d}. '{token:15s}' : {value:+.4f} {direction}")
    
    # Visualize with SHAP text plot
    print(f"\n📊 SHAP Text Visualization:")
    print(f"   (Red = pushes toward AI, Blue = pushes toward Human)\n")
    shap.plots.text(shap_values, display=True)

## 10. Error Analysis - Full Test Set

Run predictions on entire test set to find misclassifications

In [ ]:
print("\n" + "=" * 80)
print("ERROR ANALYSIS: FULL TEST SET")
print("=" * 80)

# Get predictions for entire test set
print(f"\n🔮 Running predictions on {len(test_df)} test samples...")

all_texts = test_df['text'].tolist()
y_true = test_df['binary_label'].values

# Batch prediction
batch_size = 32
y_pred_probs = []

for i in tqdm(range(0, len(all_texts), batch_size), desc="Predicting"):
    batch_texts = all_texts[i:i+batch_size]
    batch_probs = predict_fn(batch_texts)
    y_pred_probs.extend(batch_probs)

y_pred_probs = np.array(y_pred_probs)
y_pred = (y_pred_probs > 0.5).astype(int)

# Calculate accuracy
accuracy = (y_pred == y_true).mean()
print(f"\n📊 Test Set Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()

print(f"\n🎯 Confusion Matrix:")
print(f"           Predicted")
print(f"           Human  AI")
print(f"Actual Human  {tn:4d}  {fp:4d}")
print(f"       AI     {fn:4d}  {tp:4d}")

print(f"\n📋 Error Summary:")
print(f"   False Positives (Human → AI): {fp}")
print(f"   False Negatives (AI → Human): {fn}")

## 11. Analyze False Positives

Humans misclassified as AI - Did they sound robotic?

In [ ]:
print("\n" + "=" * 80)
print("FALSE POSITIVES: Humans Misclassified as AI")
print("=" * 80)

# Find false positives
fp_indices = np.where((y_true == 0) & (y_pred == 1))[0]

if len(fp_indices) > 0:
    num_fp_show = min(NUM_ERROR_SAMPLES, len(fp_indices))
    
    # Sort by confidence (most confident mistakes)
    fp_sorted = sorted(
        [(idx, y_pred_probs[idx]) for idx in fp_indices],
        key=lambda x: x[1],
        reverse=True
    )
    
    print(f"\n🔍 Showing top {num_fp_show} most confident false positives:\n")
    
    for i, (idx, confidence) in enumerate(fp_sorted[:num_fp_show]):
        text = all_texts[idx]
        label = test_df.iloc[idx]['label']
        
        print(f"{'─' * 80}")
        print(f"FALSE POSITIVE #{i+1}")
        print(f"{'─' * 80}")
        print(f"True Label: {label} (Human)")
        print(f"Predicted: AI")
        print(f"Confidence: {confidence:.4f} (AI probability)")
        print(f"\n📝 Text:")
        print(f"{text[:500]}{'...' if len(text) > 500 else ''}")
        
        # Check for robotic markers
        robotic_markers = [
            'delve', 'tapestry', 'intricate', 'nuanced', 'multifaceted',
            'paramount', 'pivotal', 'underscores', 'encompasses', 'embodies',
            'facilitates', 'necessitates', 'epitomizes', 'exemplifies'
        ]
        
        found_markers = [marker for marker in robotic_markers if marker.lower() in text.lower()]
        if found_markers:
            print(f"\n⚠️  Potential AI markers found: {', '.join(found_markers)}")
        print()
else:
    print("\n✅ No false positives! All humans correctly identified.")

## 12. Analyze False Negatives

AI texts misclassified as Human - Did they mimic well?

In [ ]:
print("\n" + "=" * 80)
print("FALSE NEGATIVES: AI Misclassified as Human")
print("=" * 80)

# Find false negatives
fn_indices = np.where((y_true == 1) & (y_pred == 0))[0]

if len(fn_indices) > 0:
    num_fn_show = min(NUM_ERROR_SAMPLES, len(fn_indices))
    
    # Sort by confidence (lowest AI probability = most confident it's Human)
    fn_sorted = sorted(
        [(idx, y_pred_probs[idx]) for idx in fn_indices],
        key=lambda x: x[1]
    )
    
    print(f"\n🔍 Showing top {num_fn_show} most confident false negatives:\n")
    
    for i, (idx, confidence) in enumerate(fn_sorted[:num_fn_show]):
        text = all_texts[idx]
        label = test_df.iloc[idx]['label']
        
        print(f"{'─' * 80}")
        print(f"FALSE NEGATIVE #{i+1}")
        print(f"{'─' * 80}")
        print(f"True Label: {label} (AI)")
        print(f"Predicted: Human")
        print(f"Confidence: {confidence:.4f} (AI probability - LOW means confident it's Human)")
        print(f"\n📝 Text:")
        print(f"{text[:500]}{'...' if len(text) > 500 else ''}")
        
        # Check if it's a mimic
        if 'Doyle' in label or 'Stevenson' in label:
            print(f"\n🎭 NOTE: This is a MIMIC sample attempting to replicate {label}'s style!")
        print()
else:
    print("\n✅ No false negatives! All AI correctly identified.")

## 13. Save Results

In [ ]:
import os

print("\n" + "=" * 80)
print("SAVING RESULTS")
print("=" * 80)

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save error analysis
error_df = pd.DataFrame({
    'text': all_texts,
    'true_label': y_true,
    'predicted_label': y_pred,
    'ai_probability': y_pred_probs,
    'original_label': test_df['label'].values,
    'is_fp': (y_true == 0) & (y_pred == 1),
    'is_fn': (y_true == 1) & (y_pred == 0)
})

error_df.to_csv(f"{OUTPUT_DIR}/error_analysis.csv", index=False)
print(f"\n💾 Saved error analysis to: {OUTPUT_DIR}/error_analysis.csv")

# Save summary
summary = {
    'test_accuracy': accuracy,
    'total_samples': len(y_true),
    'true_negatives': int(tn),
    'false_positives': int(fp),
    'false_negatives': int(fn),
    'true_positives': int(tp)
}

summary_df = pd.DataFrame([summary])
summary_df.to_csv(f"{OUTPUT_DIR}/summary.csv", index=False)
print(f"💾 Saved summary to: {OUTPUT_DIR}/summary.csv")

print("\n✅ All results saved!")

## 14. Summary & Insights

### Key Findings:

1. **SHAP Analysis** reveals which words contribute most to AI detection
2. **Common AI Markers** include:
   - Formal/academic words: "delve", "tapestry", "intricate"
   - Hedge phrases: "nuanced", "multifaceted"
   - Connector words: "furthermore", "moreover"

3. **False Positives** (Human → AI):
   - Often formal/academic human writing
   - May contain AI-like vocabulary
   - Shows model bias toward formality = AI

4. **False Negatives** (AI → Human):
   - Successfully natural-sounding AI
   - Mimic samples that replicate human style
   - Shows sophistication of modern AI

### Why This Matters:

- **Interpretability**: Know **why** the model makes decisions
- **Trust**: Validate model reasoning with human intuition
- **Debugging**: Identify biases and failure modes
- **Insights**: Discover linguistic patterns in AI text

### Next Steps:

1. Use insights to improve feature engineering (Tier A)
2. Retrain model with focus on discovered patterns
3. Develop mitigation strategies for false positives/negatives
4. Create rule-based filters for AI markers

---

**🎯 The "Smoking Gun":** SHAP shows us that AI text often reveals itself through overly formal, academic language and specific word choices that humans rarely use in natural writing.